# 🔄 RunnableWithMessageHistory: Automatic History for Any Runnable

## Learning Objectives
In this notebook, you will learn:
1. **Wrapping a chat model** - let `RunnableWithMessageHistory` prepend the stored history to a list of messages
2. **Prompt + `MessagesPlaceholder`** - decide exactly where history goes with `input_messages_key` and `history_messages_key`
3. **What the model actually receives** - render the prompt to see the injected history
4. **Extra prompt variables** - combine history with other inputs such as a response language
5. **Choosing an approach** - when the simple wrapper is enough and when you need the placeholder

## Where this fits
`02_classic_memory_classes_and_chains.ipynb` loaded and saved memory by hand around every call. `RunnableWithMessageHistory`
does both automatically: before a call it loads the session's messages, after the call it saves the new
exchange. `04_multi_user_sessions_and_persistence.ipynb` then swaps the in-memory store for SQLite and files.

## Prerequisites
- `01_why_llms_forget.ipynb` and `02_classic_memory_classes_and_chains.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

> **⚠️ Legacy API**: `RunnableWithMessageHistory` (deprecated in `langchain-core` 1.3.3) and the chat-message-history classes (1.6.4) are removed in `langchain-core` 2.0. LangChain now recommends LangGraph persistence instead — a checkpointer
> and a `thread_id` — taught in [`01_memory_and_conversational_agent.ipynb`](06a_memory_and_conversational_agent.ipynb)
> (module 02). This module keeps the older API because existing LCEL code uses it everywhere; the setup cell
> lets its deprecation warnings through so you see them where the deprecated classes are used.


---
## 📦 Step 1: Environment Setup

The setup cell loads the API keys from the project-root `.env` and silences library warnings — except LangChain's deprecation warnings, which it lets
through because this notebook uses deprecated APIs on purpose.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv
from langchain_core._api import LangChainDeprecationWarning

warnings.filterwarnings("ignore")
# This notebook uses deprecated LangChain APIs on purpose: let their deprecation warnings through
warnings.filterwarnings("default", category=LangChainDeprecationWarning)

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: the chat model wrapped with message history
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🎁 Part 1: Wrapping a Chat Model

`RunnableWithMessageHistory(runnable, get_session_history)` needs two things:
- **How to store and load messages**: a function that takes a `session_id` and returns a chat message history
- **What it wraps**: here the chat model itself, which accepts a list of messages

With no key arguments, the wrapper prepends the session's stored messages to the messages you pass in.

### Key Concepts:
- **`get_session_history(session_id)`**: returns (or creates) the history for one conversation
- **`config={"configurable": {"session_id": ...}}`**: required on every call; selects the history

In [ ]:
# ============================================================================
# PART 1A: THE SESSION STORE AND THE WRAPPER
# ============================================================================
from langchain_core.chat_history import BaseChatMessageHistory, InMemoryChatMessageHistory
from langchain_core.messages import HumanMessage
from langchain_core.runnables.history import RunnableWithMessageHistory

store: dict[str, InMemoryChatMessageHistory] = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    """Return the history for this session, creating it on first use."""
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


model_with_history = RunnableWithMessageHistory(llm, get_session_history)
print("✅ Chat model wrapped with message history")

In [ ]:
# ============================================================================
# PART 1B: TWO TURNS IN THE SAME SESSION
# ============================================================================
config = {"configurable": {"session_id": "chat1"}}

# Input can be a list of messages, a single message, or a ("human", "...") tuple
first = model_with_history.invoke([HumanMessage(content="Hi, my name is Sourav and I am a Resident Solution Architect.")], config=config)
print(f"AI: {first.content}")

second = model_with_history.invoke([HumanMessage(content="What's my name and what do I do?")], config=config)
print(f"AI: {second.content}")

print(f"\nMessages stored for 'chat1': {len(store['chat1'].messages)}")

The second call only sent one new message, yet the model knew the name: the wrapper loaded the two stored
messages from session `chat1` and put them in front. After each call it saved the new human and AI messages,
so the store now holds 4.

---
## 🧭 Part 2: Prompt + MessagesPlaceholder

Most real chains start with a prompt: a system message, the history, then the user's question. For a chain
that takes a **dict**, tell the wrapper which keys to use:
- **`input_messages_key`**: the input key holding the user's new message (saved to history after the call)
- **`history_messages_key`**: the prompt variable (`MessagesPlaceholder`) that receives the stored history

In [ ]:
# ============================================================================
# PART 2: EXPLICIT HISTORY PLACEMENT
# ============================================================================
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder

prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Always be concise and friendly."),
    MessagesPlaceholder(variable_name="history"),  # stored history is injected HERE
    ("human", "{query}"),                          # the current question
])
chain = prompt | llm | StrOutputParser()

chain_with_history = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="query",      # matches "{query}" in the prompt
    history_messages_key="history",  # matches MessagesPlaceholder(variable_name="history")
)

session = {"configurable": {"session_id": "explicit_demo"}}
for question in ["My name is Sourav and I work at Databricks.", "What's my name?", "Where do I work?"]:
    print(f"User: {question}")
    print(f"AI:   {chain_with_history.invoke({'query': question}, config=session)}\n")

---
## 🔬 Part 3: What the Model Actually Receives

Rendering the prompt with the stored history shows the exact message list sent on the next call: the system
message first, then the history at the placeholder, then the new question.

In [ ]:
# ============================================================================
# PART 3: RENDER THE PROMPT WITH THE STORED HISTORY
# ============================================================================
rendered = prompt.invoke({
    "history": store["explicit_demo"].messages,
    "query": "Tell me everything you know about me.",
})

for message in rendered.to_messages():
    print(f"{type(message).__name__:>13}: {message.content[:80]}")

![How MessagesPlaceholder works](../assets/img/image2.png)

A new `session_id` starts with an empty history — the same chain has no idea who you are there:

In [ ]:
# ============================================================================
# PART 3B: A NEW SESSION STARTS EMPTY
# ============================================================================
print(chain_with_history.invoke({"query": "What is my name?"}, config={"configurable": {"session_id": "new_session"}}))

---
## 🌐 Part 4: History Plus Other Prompt Variables

A prompt can take other inputs next to the history, for example the language to answer in. Here the
placeholder is called `messages` and receives both the history and the new message, so only
`input_messages_key="messages"` is needed.

In [ ]:
# ============================================================================
# PART 4: A SYSTEM PROMPT WITH A {language} VARIABLE
# ============================================================================
language_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant. Answer all questions to the best of your ability in {language}."),
    MessagesPlaceholder(variable_name="messages"),
])

language_chain = RunnableWithMessageHistory(
    language_prompt | llm,
    get_session_history,
    input_messages_key="messages",  # history and the new message share this placeholder
)

bengali = {"configurable": {"session_id": "chat_bengali"}}
language_chain.invoke({"messages": [HumanMessage(content="Hi, I am Krish")], "language": "Bengali"}, config=bengali)
reply = language_chain.invoke({"messages": [HumanMessage(content="What's my name?")], "language": "Bengali"}, config=bengali)
print(reply.content)

The answer should contain the name (from history) and be written in Bengali (from the system prompt variable).
The same pattern lets you place few-shot examples and history in separate placeholders:

```python
ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant"),
    MessagesPlaceholder(variable_name="examples"),  # few-shot examples first
    MessagesPlaceholder(variable_name="history"),   # then the conversation
    ("human", "{query}"),
])
```

---
## ⚖️ Part 5: Choosing an Approach

![Chat message memory flow](../assets/img/image1.png)

| Aspect | Wrap the model (Part 1) | Prompt + `MessagesPlaceholder` (Part 2) |
|---|---|---|
| Input | a list of messages | a dict with named keys |
| History placement | prepended to the input messages | exactly where the placeholder sits |
| System prompt | must be passed in the messages yourself | part of the template |
| Multiple inputs / placeholders | not possible | yes (language, examples, …) |
| Configuration | minimal | `input_messages_key` + `history_messages_key` |

For a quick prototype, wrapping the model is enough. As soon as you need a system prompt, extra variables or
control over message order, use the placeholder.

---
## 📝 Summary

In this notebook, we learned:

### 1. Wrapping a chat model
- `RunnableWithMessageHistory(llm, get_session_history)` prepends stored history and saves every exchange

### 2. Prompt + MessagesPlaceholder
- `input_messages_key` names the new message, `history_messages_key` names the placeholder

### 3. What the model receives
- System message → history at the placeholder → new question; a new `session_id` starts empty

### 4. Extra prompt variables
- History combines freely with other inputs such as a response language

### 5. Choosing an approach
- Wrap the model for prototypes; use the placeholder for real prompts

### Next Steps
- **`04_multi_user_sessions_and_persistence.ipynb`**: many users, and histories that survive a restart